In [1]:
import numpy as np
import math

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import Diagonal
from qiskit_aer import AerSimulator

In [2]:
# --------------------------------------------------
# 1. INITIALIZE THE 3-PUZZLE
# --------------------------------------------------

initial_state = [1, 2, 0, 3]

goal_state = [1, 2, 3, 0]

# 0 = UP, 1 = DOWN, 2 = LEFT, 3 = RIGHT

moves = {
    0: "UP",
    1: "DOWN",
    2: "LEFT",
    3: "RIGHT"
}

In [3]:
# --------------------------------------------------
# 2. CLASSICAL PUZZLE SIMULATOR
# --------------------------------------------------

def apply_move(state, move):

    state = state.copy()

    blank = state.index(0)

    row = blank // 2
    col = blank % 2

    if move == 0 and row > 0:
        new_position = blank - 2

    elif move == 1 and row < 1:
        new_position = blank + 2

    elif move == 2 and col > 0:
        new_position = blank - 1

    elif move == 3 and col < 1:
        new_position = blank + 1

    else:
        return state

    state[blank], state[new_position] = (
        state[new_position],
        state[blank]
    )

    return state



In [4]:
# --------------------------------------------------
# 3. VERIFY A MOVEMENT SEQUENCE
# --------------------------------------------------

def check_solution(sequence):

    state = initial_state.copy()

    for move in sequence:

        state = apply_move(state, move)

    return state == goal_state

In [5]:
# --------------------------------------------------
# 4. GENERATE ALL POSSIBLE RULE SEQUENCES
# --------------------------------------------------

depth = 2

number_of_sequences = 4 ** depth

marked_sequences = []

for sequence_number in range(number_of_sequences):

    sequence = []

    for i in range(depth):

        move = (sequence_number >> (2 * i)) & 3

        sequence.append(move)

    if check_solution(sequence):

        marked_sequences.append(sequence_number)


print("Initial State:", initial_state)
print("Goal State:", goal_state)

print("\nTotal Rule Sequences:", number_of_sequences)

print("Successful Sequences:", marked_sequences)


Initial State: [1, 2, 0, 3]
Goal State: [1, 2, 3, 0]

Total Rule Sequences: 16
Successful Sequences: [7, 13, 14, 15]


In [6]:
# --------------------------------------------------
# 5. CALCULATE GROVER ITERATIONS
# --------------------------------------------------

N = number_of_sequences
M = len(marked_sequences)

if M == 0:

    raise ValueError("No solution found at this depth.")

theta = math.asin(math.sqrt(M / N))

iterations = max(
    0,
    round(math.pi / (4 * theta) - 0.5)
)

print("\nGrover Iterations:", iterations)



Grover Iterations: 1


In [7]:
# --------------------------------------------------
# 6. CREATE QUANTUM ORACLE
# --------------------------------------------------

rule_qubits = 2 * depth

qc = QuantumCircuit(rule_qubits, rule_qubits)

# Create uniform superposition

qc.h(range(rule_qubits))


# Oracle phase vector

phase_vector = np.ones(N)

for sequence_number in marked_sequences:

    phase_vector[sequence_number] = -1


# Apply phase oracle

oracle = Diagonal(phase_vector)

qc.append(oracle, range(rule_qubits))

C:\Users\Piyush\AppData\Local\Temp\ipykernel_28088\4205967851.py:25: DeprecationWarning: The class ``qiskit.circuit.library.generalized_gates.diagonal.Diagonal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use DiagonalGate instead.
  oracle = Diagonal(phase_vector)


In [8]:
# --------------------------------------------------
# 7. GROVER DIFFUSION OPERATOR
# --------------------------------------------------

def diffusion_operator(qc, qubits):

    n = len(qubits)

    qc.h(qubits)
    qc.x(qubits)

    # Multi-controlled Z operation

    qc.h(qubits[-1])

    qc.mcx(
        qubits[:-1],
        qubits[-1]
    )

    qc.h(qubits[-1])

    qc.x(qubits)
    qc.h(qubits)


In [9]:
# --------------------------------------------------
# 8. APPLY AMPLITUDE AMPLIFICATION
# --------------------------------------------------

for _ in range(iterations):

    # Oracle

    qc.append(oracle, range(rule_qubits))

    # Diffusion

    diffusion_operator(
        qc,
        list(range(rule_qubits))
    )


In [10]:
# --------------------------------------------------
# 9. MEASUREMENT
# --------------------------------------------------

qc.measure(
    range(rule_qubits),
    range(rule_qubits)
)

In [11]:
# --------------------------------------------------
# 10. EXECUTE QUANTUM CIRCUIT
# --------------------------------------------------

simulator = AerSimulator()

compiled_circuit = transpile(
    qc,
    simulator
)

job = simulator.run(
    compiled_circuit,
    shots=2000
)

result = job.result()

counts = result.get_counts()



In [12]:
# --------------------------------------------------
# 11. DECODE THE RESULT
# --------------------------------------------------

print("\nMeasurement Results:")

print(counts)

best_result = max(
    counts,
    key=counts.get
)

sequence_number = int(best_result, 2)

solution = []

for i in range(depth):

    move = (sequence_number >> (2 * i)) & 3

    solution.append(move)



Measurement Results:
{'0101': 135, '1110': 125, '0000': 131, '1001': 142, '1010': 113, '0111': 118, '0011': 129, '1111': 114, '1000': 139, '1011': 132, '1100': 124, '0110': 136, '0100': 101, '0001': 111, '0010': 127, '1101': 123}


In [13]:
# --------------------------------------------------
# 12. DISPLAY THE SOLUTION
# --------------------------------------------------

print("\nQuantum Search Result:")

print("Encoded Sequence:", best_result)

print("Movement Sequence:")

for move in solution:

    print(moves[move], end=" ")

print()


# Verify the measured sequence

if check_solution(solution):

    print("\nSolution Successfully Verified!")

else:

    print("\nMeasured Sequence is Not a Solution.")


# Display the quantum circuit

print("\nQuantum Circuit:")

print(qc.draw())



Quantum Search Result:
Encoded Sequence: 1001
Movement Sequence:
DOWN LEFT 

Measured Sequence is Not a Solution.

Quantum Circuit:
     ┌───┐┌───────────┐┌───────────┐┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   »
q_0: ┤ H ├┤0          ├┤0          ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───»
     ├───┤│           ││           │├───┤├───┤       │  ├───┤├───┤     └╥┘┌─┐»
q_1: ┤ H ├┤1          ├┤1          ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├──────╫─┤M├»
     ├───┤│  Diagonal ││  Diagonal │├───┤├───┤       │  ├───┤├───┤      ║ └╥┘»
q_2: ┤ H ├┤2          ├┤2          ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├──────╫──╫─»
     ├───┤│           ││           │├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐ ║  ║ »
q_3: ┤ H ├┤3          ├┤3          ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫──╫─»
     └───┘└───────────┘└───────────┘└───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║  ║ »
c: 4/═══════════════════════════════════════════════════════════════════╩══╩═»
                                                                        0  1 